# NumPy Foundations for Machine Learning

Foundational machine-learning exercises implemented with NumPy, covering affine transformations, ridge regression, multiclass softmax classification, and principal component analysis (PCA).

> Portfolio-ready copy of the original completed experiment. Experimental code and saved outputs are preserved; assignment-administration text has been removed.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Read first: What is a **batched affine transformation**?
A single fully‑connected layer (linear layer) computes
$$
Y = XW + \mathbf{1} \, b^\top
$$

**Shapes**
- $X \in \mathbb{R}^{N \times D}$: **N** examples (rows), **D** features each.  
- $W \in \mathbb{R}^{D \times K}$: maps D features to **K** outputs.  
- $b \in \mathbb{R}^{K}$: one bias per output unit.  
- Result $Y \in \mathbb{R}^{N \times K}$.

**Why the $\mathbf{1} b^\top$ term?**  
$\mathbf{1} \in \mathbb{R}^{N \times 1}$ is a column of ones. $\mathbf{1} b^\top$ replicates $b$ down the batch to make an $(N,K)$ matrix so it can be added to $XW$.
In NumPy, we don’t literally create $\mathbf{1}$; **broadcasting** handles this when `b` has shape `(K,)` or `(1,K)`.

**Broadcasting mental model**
- When adding arrays with shapes $(N,K)$ and $(K,)$, NumPy “virtually stretches” $(K,)$ to $(N,K)$ row‑wise without copying.

**Three equivalent vectorized ways**
- `X @ W`  (preferred operator for matrix multiply)  
- `np.dot(X, W)`  (same for 2D)  
- `np.einsum('nd,dk->nk', X, W)`  (explicit index mapping; useful for complex contractions)

**Common gotchas**
- `b` should be shape `(K,)` or `(1,K)` (but **not** `(K,1)`).  
- No loops over rows; this is a single vectorized operation.


## Exercise 1 — Batched affine map & shape sanity
Implement and verify three equivalent computations of the affine map.


In [2]:
# Given data
np.random.seed(0)
X = np.array([[2.,-1.,0.5],[0.,1.,-1.],[1.,2.,3.]])   # (3,3)
W = np.array([[1.,0.],[2.,-1.],[-1.,1.]])             # (3,2)
b = np.array([0.5, -0.5])                             # (2,)

# compute three equivalent Y's
Y1 = X @ W + b
Y2 = np.dot(X, W) + b
Y3 = np.einsum('nd,dk->nk', X, W) + b


#MY SOLUTION
# show results and verify equality with np.allclose(...)
# print(Y1); print(Y2); print(Y3)
# -----------------------
# Show results
# -----------------------
print("Y1 = X @ W + b:\n", Y1)
print("\nY2 = np.dot(X, W) + b:\n", Y2)
print("\nY3 = np.einsum('nd,dk->nk', X, W) + b:\n", Y3)

# print(np.allclose(Y1, Y2), np.allclose(Y1, Y3))
# -----------------------
# Verify equivalence
# -----------------------
print("\nY1 == Y2 ?", np.allclose(Y1, Y2))
print("Y1 == Y3 ?", np.allclose(Y1, Y3))

Y1 = X @ W + b:
 [[ 0.   1. ]
 [ 3.5 -2.5]
 [ 2.5  0.5]]

Y2 = np.dot(X, W) + b:
 [[ 0.   1. ]
 [ 3.5 -2.5]
 [ 2.5  0.5]]

Y3 = np.einsum('nd,dk->nk', X, W) + b:
 [[ 0.   1. ]
 [ 3.5 -2.5]
 [ 2.5  0.5]]

Y1 == Y2 ? True
Y1 == Y3 ? True


## Read first: Linear regression & **ridge** (L2) regularization
We’ll fit a linear model $\hat{y} = Xw$ using the **closed-form** (normal equation). Ridge adds an L2 penalty to reduce overfitting and improve numerical conditioning.

**Standardization (why?)**
- Features often live on different scales; standardizing to mean $0$ and std $1$ helps conditioning and makes coefficients comparable.

**Add a bias term**
- Instead of handling $b$ separately, append a ones column to $X$:  
  $\tilde{X} = [X \;\; \mathbf{1}] \in \mathbb{R}^{N \times (D+1)}$, and solve for $\tilde{w}$ (last entry is bias).

**Closed-form ridge solution**
$$
w_\lambda = (X^\top X + \lambda I)^{-1} X^\top y
$$
- Use `np.linalg.solve(A, b)` for stability rather than explicit inverse.  
- Evaluate fit with **MSE**: $\frac{1}{N}\sum_{i=1}^N (\hat{y}_i - y_i)^2$.

**Interpreting $\lambda$**
- $\lambda=0$: ordinary least squares (may overfit / be ill‑conditioned).  
- Larger $\lambda$: coefficients shrink toward $0$ $\rightarrow$ lower variance, slightly higher bias.

**Checklist**
1. Standardize $X_{\text{raw}} \to X_s$.  
2. Add bias column $\to X$.  
3. For $\lambda \in \{0, 0.1, 1.0\}$, compute $w_\lambda$, predictions, MSE.  
4. Compare weights/MSE across $\lambda$.


## Exercise 2 — Ridge Linear Regression in closed form
Compute $w_\lambda$ for multiple $\lambda$ values and report MSEs.


In [3]:

# Data
X_raw = np.array([
    [3, 70, 20],
    [2, 50, 40],
    [4, 90, 15],
    [3, 60, 30],
    [5,120, 10],
    [1, 45, 50],
    [2, 55, 35],
    [4,100, 12]
], dtype=float)
y = np.array([210, 160, 280, 200, 350, 120, 170, 320], dtype=float)  # price in $k

N, D = X_raw.shape
# # 1) Standardize features -> Xs

X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0, ddof=0)
Xs = (X_raw - X_mean) / X_std

# 2) Add bias column -> X

ones = np.ones((N, 1))
X = np.hstack([Xs, ones])   # shape (N, D+1)

# Helper: MSE
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

# 3) For lambdas = [0.0, 0.1, 1.0], compute ridge weights using solve, make predictions, compute MSE

lambdas = [0.0, 0.1, 1.0]
results = []

for lam in lambdas:
    Dp1 = D + 1
    reg = np.eye(Dp1)
    reg[-1, -1] = 0.0   # do not regularize bias
    A = X.T @ X + lam * reg
    rhs = X.T @ y
    w_hat = np.linalg.solve(A, rhs)  # solve for weights
    y_pred = X @ w_hat
    results.append((lam, w_hat, mse(y, y_pred)))

# 4) Print a small table of lambda, weights, MSE

print("Lambda | Weights (standardized features..., bias) | MSE")
print("--------------------------------------------------------")
for lam, w, m in results:
    print(f"{lam:<6} | {w} | {m:.4f}")

Lambda | Weights (standardized features..., bias) | MSE
--------------------------------------------------------
0.0    | [ 17.2547  49.8052 -10.0299 226.25  ] | 74.2250
0.1    | [ 19.5301  45.8549 -11.336  226.25  ] | 75.7487
1.0    | [ 22.9004  33.2569 -17.629  226.25  ] | 110.1113


## Read first: Softmax regression (multiclass logistic)
We’ll classify $C$ classes from $D$ features. Compute **logits** $Z = X_b W$ with $X_b = [X \;\; \mathbf{1}]$, then convert logits to probabilities with **softmax**:
$$
\mathrm{softmax}(z)_k=\frac{e^{z_k}}{\sum_j e^{z_j}}
$$

**Stability trick**: subtract the row‑wise max before `exp` to avoid overflow.

**Loss: cross‑entropy** (for one‑hot labels $Y$)
$$
\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N\sum_{k=1}^C Y_{ik}\log P_{ik}
$$

**Key gradient (no loops)**
$$
\nabla_W = \frac{1}{N}\, X_b^\top (P - Y)
$$
Then a single **gradient‑descent** step: $W \leftarrow W - \eta \, \nabla_W$.

**What to report**
- Loss and accuracy **before** and **after** one step (e.g., $\eta=0.5$).

**Shape sanity**
- `Xb`: $(N, D{+}1)$; `W`: $(D{+}1, C)$; `Z, P, Y`: $(N, C)$.


## Exercise 3 — Softmax classifier: forward, loss, 1 GD step
Implement softmax, cross‑entropy, the analytic gradient, and one update.


In [4]:
# Synthetic 3-class dataset (3 Gaussian blobs)
np.random.seed(1)
N_per = 50; C = 3
means = np.array([[0,0],[2.5,2.5],[-2.5,2.5]], float)
X = np.vstack([np.random.randn(N_per,2)+m for m in means])  # (150,2)
y = np.repeat(np.arange(C), N_per)                          # (150,)
Y = np.eye(C)[y]                                            # one-hot (150,3)

# Add bias feature
Xb = np.hstack([X, np.ones((X.shape[0],1))])                # (150,3)
W = 0.01*np.random.randn(Xb.shape[1], C)                    # (3,3)

# # 1) Implement softmax with stabilization

def softmax(Z):
    Z_stable = Z - Z.max(axis=1, keepdims=True)   # subtract row max for stability
    expZ = np.exp(Z_stable)
    return expZ / expZ.sum(axis=1, keepdims=True)

# 2) Compute mean cross-entropy loss

def cross_entropy(P, Y):
    # add small epsilon to avoid log(0)
    return -np.mean(np.sum(Y * np.log(P + 1e-12), axis=1))

# Forward pass: logits -> probabilities
Z = Xb @ W
P = softmax(Z)
loss_before = cross_entropy(P, Y)
acc_before = np.mean(np.argmax(P, axis=1) == y)

# 3) Compute grad = (1/N) * Xb.T @ (P - Y)

grad = (1 / Xb.shape[0]) * Xb.T @ (P - Y)

# 4) Do ONE GD step with eta = 0.5, recompute loss & accuracy, print both

eta = 0.5
W -= eta * grad

# Recompute loss & accuracy after update
Z_new = Xb @ W
P_new = softmax(Z_new)
loss_after = cross_entropy(P_new, Y)
acc_after = np.mean(np.argmax(P_new, axis=1) == y)

# Print results
print(f"Before update -> Loss: {loss_before:.4f}, Accuracy: {acc_before:.4f}")
print(f"After  update -> Loss: {loss_after:.4f}, Accuracy: {acc_after:.4f}")

Before update -> Loss: 1.0959, Accuracy: 0.4667
After  update -> Loss: 0.5218, Accuracy: 0.7467


## Read first: PCA via SVD
**Why PCA?** Reduce dimensionality while preserving as much variance as possible; helpful for visualization and to mitigate multicollinearity.

**Steps**
1. **Standardize** features (mean $0$, std $1$).  
2. Compute covariance $\Sigma = \frac{1}{N} X_{\text{std}}^\top X_{\text{std}}$.  
3. **Eigen/SVD**: take the top‑$k$ eigenvectors (principal components). Using `np.linalg.eigh` on $\Sigma$ is common (symmetric matrix).  
4. **Project**: $Z = X_{\text{std}} U_k$ $\to$ reduced data.  
5. **Explained variance**: fraction of total variance captured by top‑$k$.

**Shapes**
- $X_{\text{std}}: (N,D)$; $\Sigma: (D,D)$; $U_k: (D,k)$; $Z: (N,k)$.


## Exercise 4 — PCA via SVD
Compute the top‑2 principal components and project the data.


In [5]:
# Reuse X_raw from Exercise 2
# # a) Standardize X_raw -> X_std

X_mean = X_raw.mean(axis=0)
X_stddev = X_raw.std(axis=0, ddof=0)   # population std
X_std = (X_raw - X_mean) / X_stddev    # standardized (N,D)

# b) Compute covariance Sigma = (1/N) * X_std.T @ X_std

N = X_std.shape[0]
Sigma = (1/N) * X_std.T @ X_std        # (D,D)

# c) Use eigh or SVD to get top-2 components U2

# eigh is for symmetric matrices (like covariance)
eigvals, eigvecs = np.linalg.eigh(Sigma)

# Sort eigenvalues (and eigenvectors) in descending order
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# Take top-2 eigenvectors
U2 = eigvecs[:, :2]   # (D,2)

# d) Project Z = X_std @ U2; print shapes and (optionally) explained variance

Z = X_std @ U2        # (N,2)

# ---------- Explained variance ----------
explained_var_ratio = eigvals[:2] / eigvals.sum()

# ---------- Results ----------
print("Shapes:")
print("X_std:", X_std.shape)
print("Sigma:", Sigma.shape)
print("U2:", U2.shape)
print("Z:", Z.shape)
print(f'Explained variance by top-2 PCs: {explained_var_ratio.sum():.4f}')
print('The U2 is:\n', U2)
print('The Z output is:\n', Z)

Shapes:
X_std: (8, 3)
Sigma: (3, 3)
U2: (3, 2)
Z: (8, 2)
Explained variance by top-2 PCs: 0.9906
The U2 is:
 [[-0.5825 -0.0537]
 [-0.5741  0.7335]
 [ 0.5754  0.6776]]
The Z output is:
 [[-0.1895 -0.4339]
 [ 1.5909  0.023 ]
 [-1.3345 -0.1425]
 [ 0.4629 -0.2273]
 [-2.7084  0.4412]
 [ 2.6045  0.4196]
 [ 1.2647 -0.0802]
 [-1.6904  0.0002]]
